# Profiling

Profiling helps you measure and optimize the performance of your workflows. This tutorial covers how to profile latency, token usage, and costs.

## What You'll Learn

1. Why profiling matters
2. Configuring profilers
3. Running profiling via CLI
4. Analyzing profiling results
5. Common optimization strategies

## Key Metrics

| Metric | Description | Why It Matters |
|--------|-------------|----------------|
| **Latency** | Time to complete request | User experience |
| **Token Usage** | Input/output tokens | Cost |
| **Tool Calls** | Number of tool invocations | Efficiency |
| **LLM Calls** | Number of LLM invocations | Cost, latency |


In [1]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


✅ Environment configured


## 1. Create a Workflow to Profile


In [2]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.sdk import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Workflow created


## 2. Configure Profiling

Profiling is configured using `ProfilerConfig` as part of the evaluation. Available profiler options:

| Option | Description |
|--------|-------------|
| `base_metrics` | Enable basic profiling metrics |
| `compute_llm_metrics` | Track LLM-specific performance metrics |
| `token_usage_forecast` | Forecast and track token usage |
| `workflow_runtime_forecast` | Forecast workflow runtime |
| `bottleneck_analysis` | Analyze performance bottlenecks |


In [3]:
import json

from nat.data_models.profiler import ProfilerConfig
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

# Create test dataset (using correct field names: id, question, answer)
test_data = [
    {"id": "calc_001", "question": "What is 2 + 2?", "answer": "The answer is 4."},
    {"id": "calc_002", "question": "What is 10 * 5?", "answer": "The answer is 50."},
    {"id": "time_001", "question": "What time is it?", "answer": "The current time."},
]

data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)
dataset_path = data_dir / "profiling_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(test_data, f, indent=2)

# Configure evaluation with profiler enabled
# ProfilerConfig fields enable specific profiling features
evaluation = NatEvaluation(
    output_dir=Path("./profiling_results"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    profiler=ProfilerConfig(
        base_metrics=True,              # Enable basic profiling metrics
        compute_llm_metrics=True,       # Track LLM-specific metrics
        token_usage_forecast=True,      # Forecast token usage
    ),
)

workflow.add_evaluator(evaluation)
print("✅ Profiler configured")


✅ Profiler configured


## 3. Save Configuration


In [4]:
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "profiling_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


📄 Saved to: configs/profiling_workflow.yaml


## 4. Run Profiling via Python


In [5]:
# Run profiling
await workflow.evaluate()
print("✅ Profiling complete! Check ./profiling_results for results")


Running workflow: 100%|██████████| 3/3 [00:01<00:00,  1.76it/s]
All evaluators were empty or invalid.


✅ Profiling complete! Check ./profiling_results for results


## 5. Visualize Profiling Results

Let's display the profiling results in a nice format:


In [6]:
from utils.visualization import display_profiling_dashboard

# Display the complete profiling dashboard with a single function call!
# Configurable parameters:
#   - fast_threshold_ms: Latency for "fast" status (default: 2000ms)
#   - medium_threshold_ms: Latency for "medium" vs "slow" (default: 5000ms)
#   - show_summary/show_table/show_chart: Toggle components
_ = display_profiling_dashboard(
    profiling_results_dir="./profiling_results",
    fast_threshold_ms=2000,    # Customize as needed
    medium_threshold_ms=5000,
)


📋 Profiling Details:



,ID,Question,Latency (ms),LLM Calls,Tool Calls
0,calc_001,What is 2 + 2?,599.779129,2,1
1,calc_002,What is 10 * 5?,639.362097,2,1
2,time_001,What time is it?,951.993942,2,1


In [7]:
# You can customize which components to display:
_ = display_profiling_dashboard(
    profiling_results_dir="./profiling_results",
    show_summary=False,  # Already shown above
    show_table=True,
    show_chart=False,
)


📋 Profiling Details:



,ID,Question,Latency (ms),LLM Calls,Tool Calls
0,calc_001,What is 2 + 2?,599.779129,2,1
1,calc_002,What is 10 * 5?,639.362097,2,1
2,time_001,What time is it?,951.993942,2,1


In [8]:
# The dashboard function returns the loaded data for further analysis:
data = display_profiling_dashboard(
    profiling_results_dir="./profiling_results",
    show_summary=False,
    show_table=False,
    show_chart=True,  # Show just the chart
)
if data:
    metrics = data["metrics"]
    print("\n📊 Metrics Summary:")
    print(f"   Average latency: {metrics['avg_latency_ms']:.0f}ms")
    print(f"   Total LLM calls: {metrics['total_llm_calls']}")
    print(f"   Total tool calls: {metrics['total_tool_calls']}")



📊 Metrics Summary:
   Average latency: 730ms
   Total LLM calls: 6
   Total tool calls: 3


## 6. Run Profiling via CLI

You can also run profiling from the command line:

```bash
# Run evaluation with profiling enabled
nat eval --config_file configs/profiling_workflow.yaml

# Run multiple repetitions for statistical significance
nat eval --config_file configs/profiling_workflow.yaml --reps 5
```


## 7. Understanding Profiling Results

After profiling, you'll find results in the output directory:

```
profiling_results/
├── profiling_summary.json    # Aggregated metrics
├── profiling_details.json    # Per-request details
└── eval_results.json         # Full evaluation results
```

### Example Profiling Summary

```json
{
  "summary": {
    "total_requests": 3,
    "avg_latency_ms": 2450,
    "p50_latency_ms": 2100,
    "p95_latency_ms": 3200,
    "total_tokens": 1500,
    "avg_tokens_per_request": 500,
    "total_llm_calls": 9,
    "avg_llm_calls_per_request": 3,
    "total_tool_calls": 6,
    "avg_tool_calls_per_request": 2
  }
}
```

### Per-Request Details

```json
{
  "requests": [
    {
      "input": "What is 2 + 2?",
      "latency_ms": 2100,
      "input_tokens": 150,
      "output_tokens": 80,
      "llm_calls": 2,
      "tool_calls": 1,
      "tools_used": ["calculator.add"]
    }
  ]
}
```


## Optimization Strategies

Based on profiling results, consider these optimizations:

### High Latency
- Use faster models (e.g., `llama-3.1-8b` instead of `70b`)
- Reduce `max_tokens` if responses are shorter than limit
- Use Tool Calling agent instead of ReAct for simple tasks

### High Token Usage
- Simplify system prompts
- Filter tool descriptions to only relevant ones
- Use `include` to limit available functions

### Too Many LLM Calls
- Improve prompts to get correct answers faster
- Use ReWOO agent for multi-step tasks (plans upfront)
- Add better error handling

## Summary

✅ **ProfilerConfig** - Enable profiling in evaluation  
✅ **Metrics** - Latency, tokens, LLM calls, tool calls  
✅ **CLI** - `nat eval` runs profiling  
✅ **Results** - Summary and per-request details  

## Next Steps

- **[12_optimization.ipynb](./12_optimization.ipynb)** - Automated optimization
- **[13_observability.ipynb](./13_observability.ipynb)** - Detailed tracing
